# Original and predicted color distributions

Research question: how do the prediction hue distribution, chroma, and signed
a* bias differ from the same held-out originals? Historical artifacts are lost;
this notebook reads only a future explicitly generated evaluation report.
No predictions or evaluation cells have been executed.
Start Jupyter from the repository root using `scripts/lab.sh`.

## Published bias measurements (abstract, p. 68)

| Quantity | Published value |
| --- | --- |
| Original red-sector fraction | approximately 0.269 |
| One model's prediction red-sector fraction | approximately 0.314 |
| Fraction difference | approximately 0.045 |
| Prediction/original ratio | approximately 1.17 |
| Mean signed delta-a* | -1.29 to -0.97 |
| Median signed delta-a* | -0.54 to -0.32 |

These are transcribed historical values, not physical-lab-v1 measurements.
The abstract does not map them to individual datasets/checkpoints or supply
raw histograms. Its negative prediction-minus-target delta-a* alone is a
greenward component shift; its warm-bias interpretation also uses hue fractions.
The red-wedge settings and historical pairing/decoding cannot be matched to
current calculations. See [published results](../reports/published-results.md).
New inference is outside this revision's scope; the code below is optional.

In [ ]:
import json
from pathlib import Path

import torch

from chromatica.visualization.color_diversity import plot_histogram

ROOT = Path.cwd()
REPORT = ROOT / ".runs/food101_unet/evaluation.json"
RUN_ANALYSIS = False

The offline evaluation command records checkpoint and test-file hashes,
configuration, environment, and metric version. It decodes physical a*/b*
with `127.5 * ab_normalized - 0.5` before computing metrics.
Enable this cell only after deliberately producing a report with that command.

In [ ]:
if RUN_ANALYSIS:
    report = json.loads(REPORT.read_text(encoding="utf-8"))
    if report["status"] != "evaluated":
        msg = "A completed evaluation report is required."
        raise ValueError(msg)
    print(report["metrics"])
    print(report["red_sector"])
    plot_histogram(torch.tensor(report["hue_gt"]), title="Held-out originals")
    plot_histogram(torch.tensor(report["hue_pred"]), title="Model predictions")

## Interpretation and limits

Positive signed delta-a* indicates a shift toward red/magenta; negative values
indicate greenward bias. MAE/RMSE report error magnitude separately. Neither
signed bias nor chroma-weighted bias is evidence of increased color diversity.
Original and predicted hue counts use independent chroma eligibility, so compare
eligible counts as well as distributions. RGB gamut clipping is only a display
operation and is excluded from physical Lab metrics. No result is claimed here.